In [0]:
%sql
DROP TABLE IF EXISTS vattenfall_dev.raw.bronze_reference;

In [0]:
import yaml
from pyspark.sql import functions as F
config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]

landing_volume = config["volumes"]["landing"]
checkpoint_volume = config["volumes"]["checkpoints"]

domain = "reference"


In [0]:
landing_path = f"/Volumes/{catalog}/{raw_schema}/{landing_volume}/{domain}/"
checkpoint_path = f"/Volumes/{catalog}/{raw_schema}/{checkpoint_volume}/{domain}/"

target_schema = f"{catalog}.{raw_schema}"

print("Landing Path:", landing_path)
print("Checkpoint Path:", checkpoint_path)

In [0]:
files = dbutils.fs.ls(landing_path)

csv_files = [f.name for f in files if f.name.endswith(".csv")]

print("Found CSV files:")
for f in csv_files:
    print("-", f)

In [0]:
for file in csv_files:
    
    # remove .csv extension → table name
    table_name = file.replace(".csv", "")
    
    file_path = f"{landing_path}{file}"
    
    print(f"\nLoading file: {file}")
    
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(file_path)

    target_table = f"{catalog}.{raw_schema}.ref_{table_name}"

    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)

    print(f"Saved -> {target_table}")
    print(f"Rows: {df.count()}")

In [0]:
print("="*60)
print("BRONZE REFERENCE TABLES CREATED")
print("="*60)

for file in csv_files:
    table_name = file.replace(".csv", "")
    table = f"{catalog}.{raw_schema}.ref_{table_name}"
    
    df = spark.table(table)
    print(f"{table:<50} {df.count():>10,} rows")

In [0]:
for file in csv_files:
    table_name = file.replace(".csv", "")
    table = f"{catalog}.{raw_schema}.ref_{table_name}"
    
    print(f"\n--- {table_name} ---")
    display(spark.table(table).limit(5))